---
title: Get started
description: How to use the Executor primitive in Qiskit Runtime.

---

# Get started with the Executor primitive

{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

Similar to the [Sampler](/docs/guides/get-started-with-sampler) primitive, Executor samples output registers from quantum circuit executions, but it does not have any built in error suppression or mitigation. Instead, it's part of the [directed execution model](/docs/guides/directed-execution-model) that provides the ingredients to capture design intents on the client side, and shifts the costly generation of circuit variants to the server side. Executor follows the directives provided in circuit annotations and options, generates and binds parameter values, executes the bound circuits on the hardware, and returns the execution results and metadata. It does not make any implicit decisions for you and gives you full control and transparency.

<Admonition type="note">
The Qiskit package does not yet have a base class for the Executor primitive.
</Admonition>

## Steps to get started with Executor

There are three basic steps involved in using the Executor primitive:
1. Initialize a `QuantumProgram` with your workload.
2. Run the `QuantumProgram` on IBM&reg; backends by using the `Executor` primitive.
3. Interpret the outputs.

See [Executor examples](/docs/guides/executor-examples) for a full walkthrough.

## Run an Executor job

The following code shows how to initialize an Executor with the default options. See [Executor options](/docs/guides/executor-options) to learn about the available options.

```python
from qiskit_ibm_runtime import Executor

executor = Executor(backend)
```
Next, use the `run()` method to submit the job.

```python
job = executor.run(program)

# Retrieve the result
result = job.result()
```

The result is of type [`QuantumProgramResult`](https://qiskit.github.io/qiskit-ibm-runtime/stubs/qiskit_ibm_runtime.quantum_program.QuantumProgramResult.html#qiskit_ibm_runtime.quantum_program.QuantumProgramResult). See [Executor input and output](/docs/guides/executor-input-output) to learn about the result object.


In [2]:
from qiskit.circuit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.transpiler import generate_preset_pass_manager
import numpy as np
from samplomatic import build
from samplomatic.transpiler import generate_boxing_pass_manager

# Generate the circuit
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.h(1)

circuit.measure_all()

# # Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Transpile the circuit
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
isa_circuit = preset_pass_manager.run(circuit)

# Initialize an empty program
program = QuantumProgram(shots=25)

# Append the circuit to the program
program.append_circuit_item(isa_circuit)

# Transpile the circuit, additionally grouping gates and measurements into annotated boxes
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
preset_pass_manager.post_scheduling = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
)
boxed_circuit = preset_pass_manager.run(circuit)

# Build the template and the samplex
template, samplex = build(boxed_circuit)

# Append the template and samplex as a samplex item
program.append_samplex_item(
   template,
   samplex=samplex,
   shape=(2,),
)

# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)
job

<RuntimeJobV2('d7h4jp22khts739pa16g', 'executor')>


## Next steps

<Admonition type="tip" title="Recommendations"> 
    - Try some [Executor examples](/docs/guides/executor-examples).
    - Understand [Executor input and output](/docs/guides/executor-input-output).
    - Learn about [Executor broadcasting semantics](/docs/guides/executor-broadcasting).
</Admonition>
